# 第11课：贪吃蛇与面向对象

本笔记本是课堂讲义。每个知识点包含：理论知识、案例代码、讲解、易错点与练习。综合练习 P1 在笔记本里完成**小场地、创建对象、移动一格、撞墙死亡**，不要求在格子里写出完整可玩游戏。完整可玩版本见同目录 [snake.py](snake.py)，终端执行 `python3 snake.py`（方向键 `w/a/s/d`）。课后独立练习见 [chapter11_贪吃蛇与面向对象_课后练习.ipynb](chapter11_贪吃蛇与面向对象_课后练习.ipynb)。

本课离开游戏数据表。职务是 **class + 二维场地**，不是再分析 Steam。

## 学习目标

1. 用 `range` 生成一维下标，再用双重 `for` 建出二维列表并打印一帧。
2. 用 `grid[row][col]` 在场地上写入蛇与食物符号。
3. 说明类是模具、实例是玩具；改一只实例的属性不会改另一只。
4. 写出带 `__init__` 的小型 `Map` / `Snake` / `Food`，用方向字典计算下一格。
5. 实现撞墙死亡，并说出主循环四步：读输入 → 更新坐标 → 判定死亡 → 刷新。

## 学习知识点

| 空间 | 类与对象 | 能玩起来 |
| --- | --- | --- |
| `range` 生成下标 | 类是模具，实例是玩具 | 方向字典 `w/a/s/d` |
| 二维列表 `grid[row][col]` | 类属性 vs 实例属性 | 下一格坐标 |
| 双重 `for` 建表、打印一帧 | `__init__` 初始化 | 撞墙死亡（当堂） |
| 写入蛇与食物符号 | 打包 Map / Snake / Food | 主循环四步 |

## 基础回顾与案例提问

第3课已经用过嵌套列表：外层一行、内层一个格子。贪吃蛇的场地就是一张这样的表。第6课已经会写函数；本课把**数据和行为**收进类，让主循环保持短。

1. **R.1** `grid = [[".", ".", "."], [".", ".", "."]]` 时，`grid[0][2]` 是什么？`grid[2][0]` 会怎样？
2. **R.2** `bad = [["."] * 3] * 2` 之后执行 `bad[0][0] = "O"`，另一行会不会一起变？为什么本课要求用双重 `for` 建表？
3. **R.3** 字典 `DIRECTIONS = {"d": (0, 1), "a": (0, -1)}` 里 `"d"` 表示什么？从 `(3, 3)` 向右走一格，新坐标是多少？

**作答：** 预测：____；依据：____；验证后说明：____。

使用 Python 3；本课不读 csv。从本文件夹启动内核。后面部分可以 `from snake import Map, Snake, Food`，但前面的格子必须自己写，不要一开始就依赖导入。本课不讲继承、多态、清屏、加速或记分。不要使用 pandas。咬自己是课后内容，当堂只要求撞墙这一种死亡。


In [ ]:
# R.1–R.3: Write and verify your predictions here.


## 1. `range`：给格子编号

### 理论知识

**`range` 生成一串整数，常用来当“第几行 / 第几列”的下标。** 它不是列表本身，但可以放进 `for` 里逐个取出。

- `range(n)` 产生 `0, 1, …, n-1`，一共 n 个，适合“有 n 列”的场地。
- `range(start, stop)` 从 start 数到 stop 之前，不含 stop。
- 循环变量只是当前这个整数；每一圈它会变成下一个。

打印场地、给蛇身编号，都先会“从 0 数到 width-1”。

### 案例：打印 range


In [1]:
print("range(5):")
for i in range(5):
    print(i)

print("range(3, 6):")
for i in range(3, 6):
    print(i)

print("3 行 5 列的行号与列号")
for r in range(3):
    for c in range(5):
        print(r, c)


range(5):
0
1
2
3
4
range(3, 6):
3
4
5
3 行 5 列的行号与列号
0 0
0 1
0 2
0 3
0 4
1 0
1 1
1 2
1 3
1 4
2 0
2 1
2 2
2 3
2 4


### 讲解

`range(5)` 得到 0 到 4，没有 5。双重循环里，外层 `r` 先固定，内层把这一行的列号走完，再换下一行。这就是后面建二维表的骨架。

`range` 本身不是 `grid`。它只提供下标；真正存符号的是列表。

### 易错点与练习

1. **K1.1** `range(4)` 会打印哪些整数？会不会包含 4？
2. **K1.2** 要给宽度 5 的一行编号，应写 `range(5)` 还是 `range(1, 5)`？用循环打印核对。

**作答：** range(4) 的值：____；宽度 5 的写法：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 2. 二维列表：`grid[row][col]`

### 理论知识

**场地是“列表的列表”。** 外层每一个元素是一行，这一行又是一个列表，里面才是格子。

坐标约定（本课锁定）：

- 第一个下标是 **行 row**，从上往下：0 是最上一行。
- 第二个下标是 **列 col**，从左往右：0 是最左一列。
- 读写一个格子：`grid[row][col]`。

这和数学课里有时先写 x 再写 y 的习惯不同。写反了会把符号放到别的格子，甚至越界。

### 案例：3×5 的点阵，读一个格子


In [2]:
grid = [
    [".", ".", ".", ".", "."],
    [".", ".", ".", ".", "."],
    [".", ".", ".", ".", "."],
]
print("行数:", len(grid))
print("第 0 行列数:", len(grid[0]))
print("grid[1][2] =", grid[1][2])
grid[1][2] = "O"
print("改写后第 1 行:", grid[1])


行数: 3
第 0 行列数: 5
grid[1][2] = .
改写后第 1 行: ['.', '.', 'O', '.', '.']


### 讲解

`len(grid)` 是行数，`len(grid[0])` 是列数。`grid[1]` 整行仍是列表；`grid[1][2]` 才是一个符号。

手写三行只适合演示。真正的场地要用循环生成，否则 8×12 会抄到崩溃。

### 易错点与练习

1. **K2.1** `grid[0][4]` 在 3×5 场地里是哪一个角？先画草图再写。
2. **K2.2** `grid[1]` 的类型是什么？`grid[1][2]` 的类型是什么？

**作答：** 角落位置：____；两个类型：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 3. 双重 `for`：建表并打印一帧

### 理论知识

**一帧场地 = 一张二维表打印出来的样子。** 建表和打印都用双重循环，不要用“列表乘列表”抄近路。

建表步骤：

1. 准备空的外层列表 `grid = []`。
2. 外层 `for r in range(height):` 每一行先建一个空 `row = []`。
3. 内层 `for c in range(width):` 往这一行 `append(".")`。
4. 一行填完，`grid.append(row)`。

打印时不要直接 `print(grid)`（那是 Python 的列表写法）。内层把一行拼成字符串，外层打印这一行。

错误写法 `[["."] * width] * height` 会让多行共享同一个内层列表：改一格，几行一起变。

### 案例：用循环生成 3×5 并打印


In [3]:
height = 3
width = 5
grid = []
for r in range(height):
    row = []
    for c in range(width):
        row.append(".")
    grid.append(row)

print("二维表:", grid)
print("一帧:")
for r in range(height):
    line = ""
    for c in range(width):
        line = line + grid[r][c]
    print(line)


二维表: [['.', '.', '.', '.', '.'], ['.', '.', '.', '.', '.'], ['.', '.', '.', '.', '.']]
一帧:
.....
.....
.....


### 讲解

输出应是三行、每行五个点。`row` 必须在外层循环里**重新**建成 `[]`，否则每一行都会往同一个列表里追加。

本课要求显式双重 `for`，不要用列表推导式凑出同一张表。

### 易错点与练习

1. **K3.1** 若把 `row = []` 写到两个 `for` 的外面，打印出来的行数还是 3 吗？
2. **K3.2** 故障写法 `bad = [["."] * 5] * 3` 再执行 `bad[0][0] = "O"`，把你预测的 `bad` 写下来（先写，不要先跑进最终流程）。

**作答：** row 写错位置：____；共享内层的现象：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 4. 在格子上写入蛇与食物

### 理论知识

**蛇和食物都是坐标，不是另造一张表。** 先有空场地，再按坐标改符号。

本课约定：

- 食物：`"*"`
- 蛇头：`"O"`（大写）
- 蛇身：`"o"`（小写）
- 空地：`"."`

蛇身用**坐标列表**保存，例如 `[(1, 1), (1, 0)]`，第一个是头。吃食物时列表变长；若一开始只用一个坐标变量，后面无法变长。

### 案例：在 3×5 上放蛇和食物


In [4]:
height = 3
width = 5
grid = []
for r in range(height):
    row = []
    for c in range(width):
        row.append(".")
    grid.append(row)

snake_body = [(1, 1), (1, 0)]
food_row = 0
food_col = 4

grid[food_row][food_col] = "*"
for index in range(len(snake_body)):
    r, c = snake_body[index]
    if index == 0:
        grid[r][c] = "O"
    else:
        grid[r][c] = "o"

for r in range(height):
    line = ""
    for c in range(width):
        line = line + grid[r][c]
    print(line)


....*
oO...
.....


### 讲解

先画食物再画蛇，这样蛇头若碰巧与食物同格，头会盖住食物，看起来像“正要吃到”。

`snake_body[0]` 是头。`index == 0` 用 `"O"`，其余用 `"o"`。坐标必须落在 `0 ≤ row < height` 且 `0 ≤ col < width`，否则下一节的撞墙就会发生。

### 易错点与练习

1. **K4.1** 若先画蛇再画食物，而食物坐标碰巧等于蛇头，这一帧会看到什么符号？
2. **K4.2** 把蛇身改成三个坐标 `[(1, 2), (1, 1), (1, 0)]`，再打印一帧。

**作答：** 覆盖顺序：____；三格蛇的一帧：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 5. 类是模具，实例是玩具

### 理论知识

**类（class）是模具，实例（object）是用模具做出的玩具。** 模具规定这种东西有哪些数据、能做哪些动作；每做一个玩具，它有自己的一份数据。

短例子用猫：猫这种模具规定“有名字”；做出两只猫，各有各的名字。看完立刻回到蛇——蛇也是同一件事：`Snake` 是模具，你创建的那条蛇才是场上的玩具。

后面不讲继承（一种模具派生出另一种模具），也不讲多态。

### 案例：两只 Cat


In [5]:
class Cat:
    def __init__(self, name):
        self.name = name

mimi = Cat("Mimi")
huahua = Cat("Huahua")
print(mimi.name)
print(huahua.name)
print(type(mimi))
print(mimi is huahua)


Mimi
Huahua
<class '__main__.Cat'>
False


### 讲解

`class Cat:` 定义模具。`Cat("Mimi")` 做出一只实例并交给 `mimi`。`huahua` 是另一只。`mimi is huahua` 为 False：两只玩具，不是同一个对象。

`self` 表示“正在做的这一只”。写 `self.name` 是给这一只贴名字，不是给模具贴名字。

### 易错点与练习

1. **K5.1** `Cat` 和 `mimi` 谁是类、谁是实例？
2. **K5.2** 再做第三只 `Cat("Coco")`，打印三只的 `name`。三只会不会共用一个名字盒子？

**作答：** 类/实例：____；第三只：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 6. 类属性与实例属性

### 理论知识

**实例属性是每一只自己的盒子；类属性是模具上共用的说明。**

- 实例属性：在 `__init__` 里写 `self.name = name`。改 `mimi.name`，`huahua.name` 不变。
- 类属性：写在 `class` 里面、方法外面，例如 `species = "cat"`。两只都可以读到同一份说明。

本课要记住的验收句：**改一只猫的名字，另一只不会跟着改。** 两条蛇也一样：改 `snake_a.body`，`snake_b.body` 不应一起变。

### 案例：改一只，另一只不动


In [6]:
class Cat:
    species = "cat"

    def __init__(self, name):
        self.name = name

mimi = Cat("Mimi")
huahua = Cat("Huahua")
print("species:", mimi.species, huahua.species)
print("before:", mimi.name, huahua.name)
mimi.name = "Xiaobai"
print("after:", mimi.name, huahua.name)


species: cat cat
before: Mimi Huahua
after: Xiaobai Huahua


### 讲解

`species` 写在类上，两只读到的都是 `"cat"`。`name` 是实例属性：`mimi.name = "Xiaobai"` 只改这一只。

如果发现改一条蛇的身体，另一条也变了，通常是把**同一个列表**传给了两条蛇。后面创建 `Snake` 时要用 `list(body)` 拷一份。

### 易错点与练习

1. **K6.1** `mimi.name = "Xiaobai"` 之后 `huahua.name` 仍是什么？这是类属性还是实例属性？
2. **K6.2** 若写出 `Cat.name = "all"`（改模具），和改 `mimi.name` 有何不同？先写判断。

**作答：** 另一只的名字：____；改类与改实例：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 7. `__init__`：做出实例时立刻填好数据

### 理论知识

**`__init__` 是构造方法：每次 `ClassName(...)` 时自动运行，用来给这一只填初始数据。**

括号里传入的参数，除了 `self`，都由你决定。场地需要高和宽，蛇需要身体坐标列表，食物需要一个格子。

`self.height = height` 的意思是：把传入的高度存进这一只场地自己的盒子。之后用 `arena.height` 读取。

### 案例：小型 Map 与 Food


In [7]:
class Map:
    def __init__(self, height, width):
        self.height = height
        self.width = width

class Food:
    def __init__(self, row, col):
        self.row = row
        self.col = col

arena = Map(3, 5)
food = Food(0, 4)
print(arena.height, arena.width)
print(food.row, food.col)


3 5
0 4


### 讲解

`Map(3, 5)` 触发 `Map.__init__`，`self` 就是正在创建的 `arena`。不必自己写 `arena.__init__(...)`。

同目录的 [snake.py](snake.py) 里也有 `Map`、`Food`、`Snake`。你可以阅读，但这一节先自己写最小版本。

### 易错点与练习

1. **K7.1** 若忘记 `self.row = row`，只写了 `row = row`，`food.row` 还能用吗？
2. **K7.2** 创建第二块场地 `Map(8, 12)`，打印两块场地的宽高，确认它们互相独立。

**作答：** 漏写 self：____；两块场地：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 8. 用类打包 Map / Snake / Food

### 理论知识

**把场地、蛇、食物各自打包，主循环就只负责发号施令。** 最小蛇需要：

- 实例属性 `body`：坐标列表，第一个是头
- 实例属性 `alive`：是否还活着
- 方法 `next_head(key)`：按方向算出下一格
- 方法 `move(...)`：走到下一格，或因撞墙把 `alive` 设为 False

方向字典可以写在类上（所有蛇共用同一份说明），身体列表必须是实例自己的。

下面是课堂简化版：先能移动和撞墙。吃食物、咬自己可以读 `snake.py`，作业再写咬自己。

### 案例：简化 Snake：打印一帧并准备移动


In [8]:
class Map:
    def __init__(self, height, width):
        self.height = height
        self.width = width

    def empty_grid(self):
        grid = []
        for r in range(self.height):
            row = []
            for c in range(self.width):
                row.append(".")
            grid.append(row)
        return grid

    def render(self, snake, food):
        grid = self.empty_grid()
        grid[food.row][food.col] = "*"
        for index in range(len(snake.body)):
            r, c = snake.body[index]
            if index == 0:
                grid[r][c] = "O"
            else:
                grid[r][c] = "o"
        for r in range(self.height):
            line = ""
            for c in range(self.width):
                line = line + grid[r][c]
            print(line)

class Food:
    def __init__(self, row, col):
        self.row = row
        self.col = col

class Snake:
    def __init__(self, body):
        self.body = list(body)
        self.alive = True

arena = Map(3, 5)
snake = Snake([(1, 1), (1, 0)])
food = Food(0, 4)
arena.render(snake, food)
print("alive:", snake.alive)


....*
oO...
.....
alive: True


### 讲解

`list(body)` 拷一份，避免外面的列表和蛇身变成同一个盒子。`render` 每次从空表画起，所以旧位置不会留下残影。

同目录 `snake.py` 的 `render` 返回字符串而不是直接 `print`，还处理吃食物。课堂简化版够用；完整游戏用终端跑 `python3 snake.py`。

### 易错点与练习

1. **K8.1** 为什么 `__init__` 里写 `self.body = list(body)`，而不是 `self.body = body`？
2. **K8.2** `arena.render(snake, food)` 里三个对象各提供什么数据？

**作答：** 拷贝列表的理由：____；三个对象的职务：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 9. 方向字典与下一格

### 理论知识

**按键只是字母，真正改变坐标的是一对 `(行变化, 列变化)`。** 本课锁定：

| 键 | 含义 | `(dr, dc)` |
| --- | --- | --- |
| `w` | 上 | `(-1, 0)` |
| `s` | 下 | `(1, 0)` |
| `a` | 左 | `(0, -1)` |
| `d` | 右 | `(0, 1)` |

头在 `(head_r, head_c)` 时，下一格是 `(head_r + dr, head_c + dc)`。

把字典写成类属性 `DIRECTIONS`，每条蛇都能用。不要用四个 `if key == "w"` 抄四遍加减。

### 案例：从当前头算出下一格


In [9]:
DIRECTIONS = {
    "w": (-1, 0),
    "s": (1, 0),
    "a": (0, -1),
    "d": (0, 1),
}

head_r = 1
head_c = 1
key = "d"
dr, dc = DIRECTIONS[key]
next_r = head_r + dr
next_c = head_c + dc
print("head:", head_r, head_c)
print("key:", key, "delta:", dr, dc)
print("next:", next_r, next_c)

key = "w"
dr, dc = DIRECTIONS[key]
print("up next:", head_r + dr, head_c + dc)


head: 1 1
key: d delta: 0 1
next: 1 2
up next: 0 1


### 讲解

向右：列 +1，行不变。向上：行 -1（因为第 0 行在屏幕上方）。

若键不在字典里，完整游戏可以选择忽略。课堂演示只用 `w/a/s/d`。

### 易错点与练习

1. **K9.1** 头在 `(0, 0)` 时按 `a`，下一格是什么？这一格还在不在 3×5 场地内？
2. **K9.2** 头在 `(1, 2)`、身体是 `[(1, 2), (1, 1), (1, 0)]`，按 `a` 的下一格是不是已经在身体里？（只判断，本课作业再写成死亡。）

**作答：** 左走出界：____；下一格是否在身体中：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 10. 撞墙死亡与主循环

### 理论知识

**当堂必做的死亡：下一格超出场地。** 条件：

`nr < 0 or nr >= height or nc < 0 or nc >= width`

成立则 `alive = False`，不要再改 `body`。咬自己（下一格已经在身体里）是课后 P1；不许回头是课后选做。

主循环的意思是：**还活着就重复**。四步：

1. 读输入（终端里是 `input`；笔记本演示改用预先写好的按键，避免格子停住等你打字）
2. 更新坐标（`move`）
3. 判定死亡（撞墙则 `alive` 为 False）
4. 刷新（再 `render` 一帧）

这是“直到死亡才停”的直觉。本课不另开 `while` 理论章；循环条件写成“还活着”。完整可玩程序在终端运行，不要在 Jupyter 里追求清屏。

### 案例：简化 Snake 走进墙；再用 snake.py 演示吃到食物


In [10]:
class Map:
    def __init__(self, height, width):
        self.height = height
        self.width = width

class Snake:
    DIRECTIONS = {
        "w": (-1, 0),
        "s": (1, 0),
        "a": (0, -1),
        "d": (0, 1),
    }

    def __init__(self, body):
        self.body = list(body)
        self.alive = True

    def next_head(self, key):
        dr, dc = self.DIRECTIONS[key]
        head_r, head_c = self.body[0]
        return head_r + dr, head_c + dc

    def move(self, key, arena):
        nr, nc = self.next_head(key)
        if nr < 0 or nr >= arena.height or nc < 0 or nc >= arena.width:
            self.alive = False
            return
        self.body.insert(0, (nr, nc))
        self.body.pop()

arena = Map(3, 5)
snake = Snake([(0, 4), (0, 3)])
print("start body:", snake.body, "alive:", snake.alive)
snake.move("d", arena)
print("after d into wall:", snake.body, "alive:", snake.alive)

print("--- 同目录 snake.py：向右走到食物 ---")
from snake import Map as GameMap, Snake as GameSnake, Food as GameFood

game_arena = GameMap(height=6, width=8)
game_snake = GameSnake([(2, 1), (2, 0)])
game_food = GameFood(2, 3)
print(game_arena.render(game_snake, game_food))
game_snake.move("d", game_food, game_arena)
game_snake.move("d", game_food, game_arena)
print("--- after eating ---")
print(game_arena.render(game_snake, game_food))
print("eaten:", game_snake.eaten, "length:", len(game_snake.body), "alive:", game_snake.alive)


start body: [(0, 4), (0, 3)] alive: True
after d into wall: [(0, 4), (0, 3)] alive: False
--- 同目录 snake.py：向右走到食物 ---
........
........
oO.*....
........
........
........
--- after eating ---
........
........
.ooO....
........
........
........
eaten: 1 length: 3 alive: True


### 讲解

简化版：头在 `(0, 4)` 再按 `d`，下一列是 5，而宽度是 5（合法列号 0–4），于是撞墙，`body` 保持原样，`alive` 变为 False。

后半段导入 `snake.py`：这是教师提供的完整类。向右两步吃到 `*`，身体变长，`eaten` 为 1。验收还可以在终端执行 `python3 snake.py --demo` 或 `python3 experiment.py`。

主循环在终端里会 `input("方向> ")`。笔记本里不要调用 `input`，否则内核会一直等待。

### 易错点与练习

1. **K10.1** 撞墙之后为什么不要 `insert` 新头？若仍然插入，蛇会画到场地外面吗？
2. **K10.2** 用自己的话写出主循环四步。哪一步在笔记本里要用预先写好的按键代替？

**作答：** 撞墙后不插入：____；四步：____。


In [ ]:
# Write your predictions, reasoning, or solution here.
# Keep deliberately faulty snippets in Markdown; run your corrected code here.


## 综合练习：小场地、对象、一步、撞墙

在本笔记本里完成，不要求写出带 `input` 的完整游戏。可以使用本节课自己写的简化类，也可以 `from snake import Map, Snake, Food`。场地建议 3×5 或 4×6。不要使用 pandas，不要写继承。

### P1.1　用双重循环建小场地并打印一帧

生成一张全是 `"."` 的二维表，打印行数、列数和一帧。必须用显式双重 `for`，不要用 `* height` 复制行。


In [ ]:
# P1.1: Build a small grid with nested for-loops and print one frame.


### P1.2　创建 Snake / Food / Map

创建场地、一条蛇、一个食物。把符号写入场地（或调用 `render`）并打印。蛇身用坐标列表，头在 `body[0]`。

**作答：** 三个对象各自保存什么：____。


In [ ]:
# P1.2: Create Map, Snake, and Food; print one frame with symbols.


### P1.3　移动一格，再演示撞墙

让蛇按一个合法方向走一格，打印新的身体。再选一个会走出边界的方向，确认 `alive` 变为 False，身体不再伸到墙外。

**作答：**

1. 合法一步后的 `body`：____
2. 撞墙用的键、撞墙后的 `alive`：____
3. 主循环四步（读输入 → …）：____


In [ ]:
# P1.3: Move one legal step, then move into a wall and show death.


## 本章总结

1. 场地是二维列表，坐标为 `grid[row][col]`；用双重 `for` 建表和打印。
2. 类是模具，实例是玩具；改一只实例的属性，另一只不变。
3. `__init__` 在创建时填数据；`Map` / `Snake` / `Food` 把数据与行为打包。
4. 方向字典给出 `(dr, dc)`，下一格 = 头 + 变化量。
5. 当堂死亡只要求撞墙。主循环：读输入 → 更新 → 判定死亡 → 刷新。完整游戏在终端运行 `python3 snake.py`。

课后请打开 [chapter11_贪吃蛇与面向对象_课后练习.ipynb](chapter11_贪吃蛇与面向对象_课后练习.ipynb)。P1 实现咬自己，P2 用两条独立的蛇，P3 选做禁止立刻反向。
